# Reproducing the kinematic / beta-recovery results

This notebook reproduces the numbers and figures in `paper/kinematic_estimation_comparison.pdf`:
per-shot kinematic parameter (theta) recovery and signal parameter (beta = (A_s, A_c)) recovery,
for four methods (Null, MAP-moments, Pixel-likelihood/"best", Oracle), on two datasets
(1e6 atoms and 1e8 atoms per shot).

There are two ways to use this notebook:

- **Option A (fast, no data/GPU needed)**: regenerate all figures and tables from the
  already-saved results in `results/*.json` (committed to this branch). This reproduces
  exactly what is in the paper.
- **Option B (full reproduction / new params)**: regenerate the results themselves from
  raw simulated data, then feed them into Option A's plotting cells. Requires the datasets
  (`download_data.sh`) and a CUDA GPU (falls back to CPU, but the pixel-likelihood method
  is very slow on CPU). Use this if you want to try different hyperparameters
  (bin count, tight-range half-width, N_RUNS, N_SHOTS, ...) for future paper iterations.

See `README.md` in the repo root for full setup instructions (environment, data download,
ais++/aispy provenance).

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT  = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()  # non_phase_shear/: results/
REPO = OUT.parent  # repo root: python-scripts/, helpers/
assert (REPO / 'python-scripts').is_dir(), f"expected to find python-scripts/ under {REPO}"
sys.path.insert(0, str(REPO / 'python-scripts'))
sys.path.insert(0, str(REPO / 'helpers'))

THETA_NAMES = ['mu_x0', 'mu_y0', 'mu_vx0', 'mu_vy0', 'sigma_x', 'sigma_y', 'sigma_vx', 'sigma_vy']
THETA_LABELS = [r'$\mu_{x0}$', r'$\mu_{y0}$', r'$\mu_{vx0}$', r'$\mu_{vy0}$',
                r'$\sigma_x$', r'$\sigma_y$', r'$\sigma_{vx}$', r'$\sigma_{vy}$']
METHODS = ['null', 'moments', 'best', 'oracle']
METHOD_LABELS = {'null': 'Null (prior only)', 'moments': 'MAP-moments (Kalman)',
                  'best': 'Pixel-likelihood (bins=32, tight range)', 'oracle': 'Oracle (true theta)'}
METHOD_COLORS = {'null': 'gray', 'moments': 'tab:orange', 'best': 'tab:blue', 'oracle': 'tab:green'}
DATASETS = ['1e6', '1e8']
DATASET_LABELS = {'1e6': r'$10^6$ atoms', '1e8': r'$10^8$ atoms'}
N_RUNS, N_SHOTS = 10, 50   # must match the run that produced results/*.json below

## Option B (optional): regenerate results from raw data

Skip this section if you just want to reproduce the existing paper numbers/plots (go to
"Option A" below) -- `results/*.json` already contains them.

To regenerate from scratch (e.g. after changing hyperparameters in the analysis scripts),
run from the repo root (not from inside the notebook -- these are long-running, GPU-bound
jobs better run as scripts so you get progress logging per run):

```bash
python analysis/generate_kinematic_estimates.py 1e6 10 50
python analysis/generate_kinematic_estimates.py 1e8 10 50
python analysis/beta_fits_from_kinematics.py 1e6 10 50
python analysis/beta_fits_from_kinematics.py 1e8 10 50
```

Arguments are `<dataset: 1e6|1e8> <n_runs> <n_shots>`. To try different hyperparameters
(bin count, tight-range half-width, Gauss-Hermite order for beta fitting, prior, etc.),
edit the module-level constants near the top of `analysis/generate_kinematic_estimates.py`
(`BINS_BEST`, `TIGHT_HALF_RANGE`, `PIXEL_NGH`, `M_PHI`) and `analysis/beta_fits_from_kinematics.py`
(`GH_ORDER`, `GH_CHUNK`, `BINS_BETA`), then re-run both scripts and re-run the cells below.

This requires the datasets to be present under `data/` -- run `./download_data.sh` first
(see README.md).

## Option A: reproduce numbers and figures from saved results

In [ ]:
def load_kinematics(dataset):
    with open(OUT / 'results' / f'kinematic_estimates_{dataset}_N{N_RUNS}_shots{N_SHOTS}.json') as f:
        return json.load(f)

def load_beta_fits(dataset):
    with open(OUT / 'results' / f'beta_fits_{dataset}_N{N_RUNS}_shots{N_SHOTS}.json') as f:
        return json.load(f)

def compute_residuals(kin_data, method):
    """Returns (n_shots_total, 8) residuals, pooling z0 and z100 and all runs/shots."""
    residuals = []
    for run_name, run in kin_data.items():
        for shot in run['shots']:
            true_z0 = np.array(shot['true_theta_z0']); est_z0 = np.array(shot[f'theta_{method}_z0'])
            true_z100 = np.array(shot['true_theta_z100']); est_z100 = np.array(shot[f'theta_{method}_z100'])
            residuals.append(est_z0 - true_z0)
            residuals.append(est_z100 - true_z100)
    return np.array(residuals)

def compute_rmse_table(kin_data, method):
    res = compute_residuals(kin_data, method)
    return np.sqrt((res**2).mean(axis=0))

### Kinematic (theta) parameter RMSE tables (mu in um, um/s; sigma in um, um/s)

In [ ]:
for dataset in DATASETS:
    kin = load_kinematics(dataset)
    print(f'\n=== {DATASET_LABELS[dataset]} ===')
    header = f'{"param":10s} ' + ' '.join(f'{m:>10s}' for m in METHODS) + '   [um or um/s]'
    print(header)
    for k in range(8):
        row = f'{THETA_NAMES[k]:10s} '
        for method in METHODS:
            rmse_um = compute_rmse_table(kin, method)[k] * 1e6
            row += f'{rmse_um:10.3f} '
        print(row)

### Beta = (A_s, A_c) RMSE table (mrad)

In [ ]:
for dataset in DATASETS:
    beta_data = load_beta_fits(dataset)
    print(f'\n=== {DATASET_LABELS[dataset]} ===')
    for method in METHODS:
        rows = beta_data[method]
        errs = np.array([[r['beta'][0]-r['As_true'], r['beta'][1]-r['Ac_true']] for r in rows])
        rmse_mrad = np.sqrt((errs**2).mean(axis=0)) * 1e3
        print(f'  {method:10s}: RMSE(As)={rmse_mrad[0]:.4f} mrad  RMSE(Ac)={rmse_mrad[1]:.4f} mrad')

### Figure: kinematic parameter residual histograms

In [ ]:
for dataset in DATASETS:
    kin = load_kinematics(dataset)
    res_by_method = {m: compute_residuals(kin, m) for m in METHODS}
    fig, axes = plt.subplots(2, 4, figsize=(20, 9))
    for k in range(8):
        ax = axes.flat[k]
        all_vals = np.concatenate([res_by_method[m][:, k] for m in ['null', 'moments', 'best']])
        lo, hi = np.percentile(all_vals, [0.5, 99.5])
        pad = 0.1 * (hi - lo)
        xlim = (lo - pad, hi + pad)
        for method in METHODS:
            res = res_by_method[method][:, k]
            in_range = res[(res >= xlim[0]) & (res <= xlim[1])]
            ax.hist(in_range, bins=50, range=xlim, histtype='step', density=True, linewidth=1.8,
                    color=METHOD_COLORS[method], label=METHOD_LABELS[method])
        ax.set_xlim(xlim)
        ax.set_title(THETA_LABELS[k], fontsize=13)
        ax.axvline(0, color='k', linewidth=0.7, linestyle=':')
        ax.set_xlabel('estimate - truth  [m or m/s]')
        ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
    axes.flat[0].legend(fontsize=8, loc='upper left')
    fig.suptitle(f'Kinematic parameter residuals, {DATASET_LABELS[dataset]}', fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

### Figure: beta = (A_s, A_c) recovery scatter

In [ ]:
for dataset in DATASETS:
    beta_data = load_beta_fits(dataset)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharex=True, sharey=True)
    for i, method in enumerate(METHODS):
        ax = axes[i]
        rows = beta_data[method]
        As_true = rows[0]['As_true']; Ac_true = rows[0]['Ac_true']
        As_hat = [r['beta'][0] for r in rows]; Ac_hat = [r['beta'][1] for r in rows]
        ax.scatter(As_hat, Ac_hat, color=METHOD_COLORS[method], s=50, alpha=0.8, zorder=3)
        ax.scatter([As_true], [Ac_true], color='red', marker='*', s=250, zorder=5, label='truth')
        ax.set_title(METHOD_LABELS[method], fontsize=11)
        ax.set_xlabel('$A_s$')
        if i == 0:
            ax.set_ylabel('$A_c$')
        ax.axhline(Ac_true, color='red', linewidth=0.5, alpha=0.3)
        ax.axvline(As_true, color='red', linewidth=0.5, alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle(f'$(\\hat A_s, \\hat A_c)$ across {N_RUNS} independent runs, {DATASET_LABELS[dataset]}', fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()

### Figure: 2D joint residual structure (scatter + KDE contours)

Reveals correlation structure between mean-pair and spread-pair residuals, per method.
See paper Discussion / Appendix for interpretation.

In [ ]:
from scipy.stats import gaussian_kde

PAIRS = [('mu_x0', 'mu_vx0'), ('mu_y0', 'mu_vy0'), ('sigma_x', 'sigma_vx'), ('sigma_y', 'sigma_vy')]
PAIR_METHODS = ['null', 'moments', 'best']  # oracle excluded: delta function at 0, uninformative in 2D
THETA_LABELS_MAP = dict(zip(THETA_NAMES, THETA_LABELS))

for dataset in DATASETS:
    kin = load_kinematics(dataset)
    res_by_method = {m: compute_residuals(kin, m) for m in PAIR_METHODS}

    fig, axes = plt.subplots(len(PAIRS), len(PAIR_METHODS), figsize=(5 * len(PAIR_METHODS), 4.5 * len(PAIRS)))
    for row, (px, py) in enumerate(PAIRS):
        ix, iy = THETA_NAMES.index(px), THETA_NAMES.index(py)
        all_x = np.concatenate([res_by_method[m][:, ix] for m in PAIR_METHODS])
        all_y = np.concatenate([res_by_method[m][:, iy] for m in PAIR_METHODS])
        xlo, xhi = np.percentile(all_x, [0.5, 99.5]); ylo, yhi = np.percentile(all_y, [0.5, 99.5])
        xpad = 0.15 * (xhi - xlo); ypad = 0.15 * (yhi - ylo)
        xlim = (xlo - xpad, xhi + xpad); ylim = (ylo - ypad, yhi + ypad)

        for col, method in enumerate(PAIR_METHODS):
            ax = axes[row, col]
            x = res_by_method[method][:, ix]; y = res_by_method[method][:, iy]
            in_range = (x >= xlim[0]) & (x <= xlim[1]) & (y >= ylim[0]) & (y <= ylim[1])
            x_r, y_r = x[in_range], y[in_range]
            ax.scatter(x_r, y_r, s=4, alpha=0.25, color=METHOD_COLORS[method])
            try:
                xy = np.vstack([x_r, y_r])
                kde = gaussian_kde(xy)
                xx, yy = np.mgrid[xlim[0]:xlim[1]:80j, ylim[0]:ylim[1]:80j]
                zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
                ax.contour(xx, yy, zz, levels=5, colors=METHOD_COLORS[method], linewidths=1.2)
            except Exception as e:
                ax.text(0.5, 0.5, f'(contour failed: {e})', transform=ax.transAxes, fontsize=7, ha='center')
            corr = np.corrcoef(x_r, y_r)[0, 1]
            ax.axhline(0, color='k', linewidth=0.5, linestyle=':')
            ax.axvline(0, color='k', linewidth=0.5, linestyle=':')
            ax.set_xlim(xlim); ax.set_ylim(ylim)
            ax.set_xlabel(f'{THETA_LABELS_MAP[px]} residual')
            ax.set_ylabel(f'{THETA_LABELS_MAP[py]} residual')
            ax.set_title(f'{METHOD_LABELS[method]}\n(Pearson r = {corr:.3f})', fontsize=10)
            ax.ticklabel_format(axis='both', style='sci', scilimits=(0, 0))
    fig.suptitle(f'Joint residual structure, {DATASET_LABELS[dataset]}', fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()